In [1]:

# ==========================================================
# BOOSTING BENCHMARK - PLACEMENT PREDICTION SYSTEM
# ==========================================================

import sys
import subprocess
import importlib
import warnings

warnings.filterwarnings("ignore")


# ==========================================================
# 1. INSTALL REQUIRED PACKAGES IF MISSING
# ==========================================================

def install_if_missing(package_name, import_name=None):

    if import_name is None:
        import_name = package_name

    try:
        importlib.import_module(import_name)
        print(package_name, "already installed.")

    except ImportError:

        print("Installing", package_name, "...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-U",
            package_name
        ])

        print(package_name, "installed successfully.")


install_if_missing("xgboost")
install_if_missing("lightgbm")


# ==========================================================
# 2. IMPORT LIBRARIES
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ==========================================================
# 3. LOAD DATASET
# ==========================================================

DATA_PATH = (
    r"C:\Users\pendi\2-1-ML\PLACEMENT PREDICTION SYSTEM"
    r"\data\placement_predict_50k Dataset_final (1).csv"
)

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ==========================================================
# 4. CHECK TARGET COLUMN
# ==========================================================

if "PlacementStatus" not in df.columns:

    raise ValueError(
        "PlacementStatus column not found in dataset."
    )


# ==========================================================
# 5. ENCODE TARGET
# ==========================================================

if df["PlacementStatus"].dtype == "object":

    df["PlacementStatus"] = (
        df["PlacementStatus"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "no": 0,
            "yes": 1
        })
    )

else:

    df["PlacementStatus"] = pd.to_numeric(
        df["PlacementStatus"],
        errors="coerce"
    )


# Remove missing target rows
df = df.dropna(
    subset=["PlacementStatus"]
)

df["PlacementStatus"] = (
    df["PlacementStatus"]
    .astype(int)
)


# ==========================================================
# 6. DEFINE FEATURES AND TARGET
# ==========================================================

drop_columns = [
    "PlacementStatus"
]

if "StudentID" in df.columns:

    drop_columns.append(
        "StudentID"
    )

X = df.drop(
    columns=drop_columns
)

y = df["PlacementStatus"]


# ==========================================================
# 7. CONVERT CATEGORICAL FEATURES
# ==========================================================

for column in X.columns:

    if X[column].dtype == "object":

        values = (
            X[column]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .unique()
        )

        # Yes / No columns
        if set(values).issubset(
            {"yes", "no"}
        ):

            X[column] = (
                X[column]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({
                    "no": 0,
                    "yes": 1
                })
            )

        else:

            # Other categorical columns
            X[column] = (
                X[column]
                .astype("category")
                .cat.codes
            )


# ==========================================================
# 8. CONVERT FEATURES TO NUMERIC
# ==========================================================

for column in X.columns:

    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )


# ==========================================================
# 9. HANDLE INFINITE AND MISSING VALUES
# ==========================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

for column in X.columns:

    if X[column].isna().any():

        median_value = X[column].median()

        if pd.isna(median_value):

            median_value = 0

        X[column] = X[column].fillna(
            median_value
        )


# ==========================================================
# 10. DISPLAY DATA INFORMATION
# ==========================================================

print("\n" + "=" * 60)
print("FEATURE INFORMATION")
print("=" * 60)

print(
    "Number of features:",
    X.shape[1]
)

print("\nFeatures:")

for feature in X.columns:

    print("-", feature)


print("\nTarget distribution:")

print(
    y.value_counts()
)


# ==========================================================
# 11. TRAIN / VALIDATION SPLIT
# ==========================================================

X_train, X_val, y_train, y_val = train_test_split(

    X,
    y,

    test_size=0.30,

    random_state=42,

    stratify=y
)


print("\n" + "=" * 60)
print("TRAIN / VALIDATION SPLIT")
print("=" * 60)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Validation samples:",
    len(X_val)
)


# ==========================================================
# 12. CREATE BOOSTING MODELS
# ==========================================================

models = {

    "AdaBoost": AdaBoostClassifier(

        n_estimators=100,

        learning_rate=0.1,

        random_state=42

    ),

    "Gradient Boosting": GradientBoostingClassifier(

        n_estimators=100,

        learning_rate=0.1,

        max_depth=3,

        random_state=42

    ),

    "XGBoost": XGBClassifier(

        n_estimators=100,

        learning_rate=0.1,

        max_depth=3,

        random_state=42,

        eval_metric="logloss",

        n_jobs=-1

    ),

    "LightGBM": LGBMClassifier(

        n_estimators=100,

        learning_rate=0.1,

        max_depth=3,

        random_state=42,

        verbosity=-1,

        n_jobs=-1

    )

}


# ==========================================================
# 13. TRAIN ALL MODELS
# ==========================================================

results = []

print("\n" + "=" * 60)
print("BOOSTING BENCHMARK")
print("=" * 60)


for name, model in models.items():

    print("\nTraining:", name)

    # Train
    model.fit(
        X_train,
        y_train
    )

    # Training prediction
    train_pred = model.predict(
        X_train
    )

    # Validation prediction
    val_pred = model.predict(
        X_val
    )

    # Calculate accuracy
    train_accuracy = accuracy_score(
        y_train,
        train_pred
    )

    val_accuracy = accuracy_score(
        y_val,
        val_pred
    )

    # Save results
    results.append({

        "Model": name,

        "Train Accuracy": train_accuracy,

        "Validation Accuracy": val_accuracy

    })

    print(
        "Train Accuracy:",
        round(
            train_accuracy * 100,
            2
        ),
        "%"
    )

    print(
        "Validation Accuracy:",
        round(
            val_accuracy * 100,
            2
        ),
        "%"
    )


# ==========================================================
# 14. CREATE RESULTS DATAFRAME
# ==========================================================

results_df = pd.DataFrame(
    results
)


# ==========================================================
# 15. SORT RESULTS
# ==========================================================

results_df = results_df.sort_values(

    by="Validation Accuracy",

    ascending=False

).reset_index(
    drop=True
)


# ==========================================================
# 16. DISPLAY FINAL RESULTS
# ==========================================================

print("\n" + "=" * 60)
print("FINAL BOOSTING BENCHMARK RESULTS")
print("=" * 60)

print(
    results_df.to_string(
        index=False
    )
)


# ==========================================================
# 17. BEST MODEL
# ==========================================================

best_model = results_df.iloc[0]


print("\n" + "=" * 60)
print("BEST MODEL")
print("=" * 60)

print(
    "Model:",
    best_model["Model"]
)

print(
    "Validation Accuracy:",
    round(
        best_model["Validation Accuracy"] * 100,
        2
    ),
    "%"
)


# ==========================================================
# 18. SAVE RESULTS
# ==========================================================

OUTPUT_PATH = (
    r"C:\Users\pendi\2-1-ML\PLACEMENT PREDICTION SYSTEM"
    r"\data\boosting_benchmark_results.csv"
)

results_df.to_csv(
    OUTPUT_PATH,
    index=False
)


print("\n" + "=" * 60)
print("RESULTS SAVED SUCCESSFULLY")
print("=" * 60)

print(
    OUTPUT_PATH
)

xgboost already installed.
lightgbm already installed.
DATASET LOADED SUCCESSFULLY
Dataset shape: (50000, 32)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly', 'Salary Package']

FEATURE INFORMATION
Number of features: 30

Features:
- Gender
- City
- CollegeTier
- Stream
- Specialisation
- Hostel
- HistoryOfBacklogs
- SGPA_Sem1
- SGPA_Sem2
- SGPA_Sem3
- SGPA_Sem4
- SGPA_Sem5
- SGPA_Sem6
- SGPA_Sem7
- SGPA_Sem8
- CGPA
- AttendancePercent
- Internships
- Projects
- Workshops
- Certifications
- Publications
- AptitudeTestScore
- SoftSkillsRating
- CodingTestScore
- MockIntervie

In [ ]:
# ============================================================
# BOOSTING + XGBOOST + SHAP
# PLACEMENT PREDICTION SYSTEM
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import shap


# ============================================================
# 1. LOAD DATASET
# ============================================================

DATA_PATH = r"C:\Users\pendi\2-1-ML\PLACEMENT PREDICTION SYSTEM\data\placement_predict_50k Dataset_final (1).csv"

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


# ============================================================
# 2. CHECK TARGET
# ============================================================

if "PlacementStatus" not in df.columns:
    raise ValueError("PlacementStatus column not found!")


# ============================================================
# 3. ENCODE TARGET
# ============================================================

if df["PlacementStatus"].dtype == "object":

    df["PlacementStatus"] = (
        df["PlacementStatus"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "no": 0,
            "yes": 1
        })
    )

else:

    df["PlacementStatus"] = pd.to_numeric(
        df["PlacementStatus"],
        errors="coerce"
    )


# Remove missing target rows
df = df.dropna(
    subset=["PlacementStatus"]
)

df["PlacementStatus"] = df[
    "PlacementStatus"
].astype(int)


# ============================================================
# 4. DEFINE X AND y
# ============================================================

drop_columns = ["PlacementStatus"]

if "StudentID" in df.columns:
    drop_columns.append("StudentID")

X = df.drop(
    columns=drop_columns
)

y = df["PlacementStatus"]


# ============================================================
# 5. CONVERT CATEGORICAL COLUMNS
# ============================================================

for column in X.columns:

    if X[column].dtype == "object":

        values = (
            X[column]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
        )

        unique_values = set(values.unique())

        # Yes / No column
        if unique_values.issubset({"yes", "no"}):

            X[column] = (
                X[column]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({
                    "no": 0,
                    "yes": 1
                })
            )

        else:

            X[column] = (
                X[column]
                .astype("category")
                .cat.codes
            )


# ============================================================
# 6. CONVERT TO NUMERIC
# ============================================================

for column in X.columns:

    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )


# ============================================================
# 7. HANDLE MISSING VALUES
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

for column in X.columns:

    if X[column].isna().any():

        median_value = X[column].median()

        if pd.isna(median_value):
            median_value = 0

        X[column] = X[column].fillna(
            median_value
        )


# ============================================================
# 8. TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(

    X,
    y,

    test_size=0.30,

    random_state=42,

    stratify=y
)


print("\n" + "=" * 60)
print("TRAIN / VALIDATION DATA")
print("=" * 60)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)


# ============================================================
# 9. CREATE XGBOOST MODEL
# ============================================================

xgb = XGBClassifier(

    n_estimators=500,

    learning_rate=0.05,

    max_depth=3,

    random_state=42,

    eval_metric="logloss",

    n_jobs=-1
)


# ============================================================
# 10. TRAIN XGBOOST
# ============================================================

print("\nTraining XGBoost...")

xgb.fit(
    X_train,
    y_train
)


# ============================================================
# 11. XGBOOST PREDICTION
# ============================================================

xgb_pred = xgb.predict(
    X_val
)

xgb_accuracy = accuracy_score(
    y_val,
    xgb_pred
)


print("\n" + "=" * 60)
print("XGBOOST RESULT")
print("=" * 60)

print(
    "Validation Accuracy:",
    round(xgb_accuracy * 100, 2),
    "%"
)


# ============================================================
# 12. SHAP EXPLAINER
# ============================================================

print("\nCreating SHAP explanation...")

explainer = shap.TreeExplainer(
    xgb
)

shap_values = explainer.shap_values(
    X_val
)


# ============================================================
# 13. SHAP BEESWARM PLOT
# ============================================================

print("Creating SHAP Beeswarm Plot...")

plt.figure(
    figsize=(10, 6)
)

shap.summary_plot(
    shap_values,
    X_val,
    plot_type="dot"
)

plt.title(
    "XGBoost SHAP Beeswarm Plot"
)

plt.show()


print("\nDONE!")